# 🌍 Project ÀṢÀ / AfriWise — Qwen-2.5-3B (Fast 45-Min Optimized Build)
### Fine-Tuning Qwen2.5-3B-Instruct for Igbo, Efik, and Edo (Bini)
**Optimizations Applied:**
- **500 Steps Max:** Finishes in ~40-45 minutes (well under Colab's 75-min timeout).
- **Drive Checkpointing:** Saves checkpoint every 100 steps directly to Google Drive so progress is never lost.
- **Full Tri-Lingual Dataset:** Ingests Igbo, Efik, and Edo data + <think> CoT reasoning.

## 📦 Step 1: Mount Drive & Install Dependencies

In [ ]:
# 1. Mount Google Drive first so checkpoints save automatically
from google.colab import drive
drive.mount('/content/drive')

print("📦 Installing Unsloth and training packages...")
# 2. Install dependencies optimized for T4 GPU
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets sentencepiece gradio

## 💬 Step 2: Master Layered System Prompts

In [ ]:
MASTER_SYSTEM_PROMPT = """You are ÀṢÀ (AfriWise) — a culturally grounded, context-aware AI assistant specializing in southern Nigerian languages (Igbo, Efik/Ibibio, Edo/Bini) and English.

## CORE CAPABILITIES & DOMAINS:
1. Educational & Cultural Guidance: You explain grammar, proverbs, oral traditions, and vocabulary accurately.
2. Health & Life-Saving Protocol: You provide structured first-aid guidance, symptom assessment, and emergency recommendations. In high-risk emergencies (difficulty breathing, severe bleeding, seizures, unconsciousness), prioritize immediate action and calling emergency numbers (112 or 199).
3. Multilingual Separation: Never mix Igbo, Efik, or Edo words together. Each is a distinct linguistic system.

## INTERNAL COGNITIVE PIPELINE:
Always think through your response in English inside <think> tags before speaking:
- Detect user language and intent.
- Retrieve verified facts (medical, historical, or linguistic) in English.
- Formulate the solution in English.
- Translate the final response into the requested language with proper tone and diacritics.

## ANTI-HALLUCINATION & HONESTY RULES:
Never invent words, fake translations, or fabricate traditions. If you are uncertain, honestly state:
- Igbo: 'A maghị m' (I do not know)
- Efik: 'Mmọdiọkke' (I do not know)
- Edo: 'I ma-ẹre' (I do not know)"""

print("✅ Master System Prompt Configured.")

## 📥 Step 3: Ingest Multi-Source Datasets (Igbo, Efik, Edo)

In [ ]:
import os
import json
import subprocess
import random
from datasets import load_dataset, Dataset

# 1. Clone Edo repository (Niger-Volta-LTI/edo-text)
print("📥 Fetching Edo data...")
if not os.path.exists("edo-text"):
    subprocess.run(["git", "clone", "--depth=1", "https://github.com/Niger-Volta-LTI/edo-text.git"])
    print("  ✅ Edo repo cloned.")
else:
    print("  ⏭️ Edo repo ready.")

# 2. Load Igbo from Masakhane
print("📥 Loading Igbo MT pairs...")
igbo_ds = load_dataset("json", data_files="https://raw.githubusercontent.com/masakhane-io/lafand-mt/main/data/json_files/en-ibo/train.json", split="train")
print(f"  ✅ Loaded {len(igbo_ds)} Igbo pairs.")

# 3. Load Efik from Davlan Ibom-MT
print("📥 Loading Efik MT pairs...")
efik_ds = load_dataset("Davlan/ibom-mt-en-efi", split="train")
print(f"  ✅ Loaded {len(efik_ds)} Efik pairs.")

# 4. Parse Edo parallel text
edo_pairs = []
if os.path.exists("edo-text"):
    for root, dirs, files in os.walk("edo-text"):
        for file in files:
            if file.endswith((".txt", ".tsv", ".json")):
                try:
                    with open(os.path.join(root, file), "r", encoding="utf-8", errors="ignore") as f:
                        for line in f:
                            if "\t" in line:
                                parts = line.strip().split("\t")
                                if len(parts) >= 2 and len(parts[0]) > 2 and len(parts[1]) > 2:
                                    edo_pairs.append({"en": parts[0].strip(), "edo": parts[1].strip()})
                except Exception:
                    pass

print(f"  ✅ Extracted {len(edo_pairs)} Edo pairs.")

## 🛠️ Step 4: Build CoT Training Dataset

In [ ]:
CURATED_COT_SAMPLES = [
    {
        "user": "Kedu ka mmadụ si agwọ ahụ ọkụ?",
        "thought": "Language: Igbo.\nQuery: How to treat a fever?\nMedical Fact: Hydration, rest, paracetamol, clinic malaria screening.\nTarget: Natural Igbo.",
        "assistant": "Iji gwọọ ahụ ọkụ:\n1. Nwee izu ike ma ṅụọ mmiri zuru oke.\n2. Ị nwere ike iji ọgwụ dịka Paracetamol belata oke ọkụ ahụ.\n3. Ọ bụrụ na ahụ ọkụ ahụ agaghị ma ọ bụ dị elu, gaa ụlọ ọgwụ ngwa ngwa ka dọkịta nyochaa gị maka ịba (malaria)."
    },
    {
        "user": "What should I do if someone is bleeding heavily?",
        "thought": "Language: English.\nMedical Emergency: Heavy bleeding.\nFirst Aid: Direct pressure, elevate wound, emergency services (112 / 199).",
        "assistant": "⚠️ EMERGENCY FIRST AID FOR HEAVY BLEEDING:\n1. Apply firm, direct pressure on the wound with a clean cloth or bandage.\n2. Do NOT remove the cloth if soaked; add more layers on top.\n3. Elevate the injured limb above heart level if possible.\n4. Call emergency services (112 / 199) or take the patient to the nearest hospital immediately.\n5. Keep the person calm and warm."
    },
    {
        "user": "What are the six vital consonant digraphs in Bini-Edo?",
        "thought": "Topic: Edo Orthography.\nFacts: The 6 distinct digraphs are GH, KH, GB, KP, VB, and MW.\nFormatting: Clear educational breakdown.",
        "assistant": "The six vital consonant digraphs in Bini-Edo orthography are:\n1. **GH** (velar fricative)\n2. **KH** (voiceless velar fricative)\n3. **GB** (voiced labial-velar stop)\n4. **KP** (voiceless labial-velar stop)\n5. **VB** (voiced labiodental approximant)\n6. **MW** (nasalized labial glide)\n\nThese digraphs represent unique phonemes essential for writing and pronouncing Edo accurately."
    },
    {
        "user": "Explain the Igbo proverb: 'Ilu bụ mmanụ e ji eri okwu'.",
        "thought": "Topic: Igbo Culture / Proverbs.\nLiteral: Proverbs are the palm oil with which words are eaten.\nMeaning: Proverbs provide eloquence, diplomacy, and clarity in Igbo discourse.",
        "assistant": "The Igbo proverb **'Ilu bụ mmanụ e ji eri okwu'** translates to *'Proverbs are the palm oil with which words are eaten.'*\n\nIn Igbo philosophy (Omenala Igbo), proverbs are considered the highest form of speech. Just as palm oil makes yam or food palatable and easy to swallow, proverbs make complex truths, diplomatic discussions, and wisdom easy to understand and digest."
    }
]

chatml_template = """<|im_start|>system
{system}<|im_end|>
<|im_start|>user
{user}<|im_end|>
<|im_start|>assistant
<think>
{thought}
</think>
{assistant}<|im_end|>"""

all_formatted_texts = []

# 1. Add curated CoT samples with high sample weighting
for _ in range(25):
    for sample in CURATED_COT_SAMPLES:
        all_formatted_texts.append({"text": chatml_template.format(system=MASTER_SYSTEM_PROMPT, user=sample["user"], thought=sample["thought"], assistant=sample["assistant"])})

# 2. Format Igbo
print("🔄 Formatting Igbo dataset...")
for item in igbo_ds["translation"][:10000]:
    en = item.get("en", "").strip()
    ibo = item.get("ibo", "").strip()
    if en and ibo:
        q = f"Translate to Igbo: {en}"
        th = f"Language: Igbo.\nIntent: Translate English text to Igbo.\nInput: '{en}'.\nSyntactic mapping to Igbo."
        all_formatted_texts.append({"text": chatml_template.format(system=MASTER_SYSTEM_PROMPT, user=q, thought=th, assistant=ibo)})

# 3. Format Efik
print("🔄 Formatting Efik dataset...")
for item in efik_ds["translation"][:8000]:
    en = item.get("en", "").strip()
    efi = item.get("ig", "").strip()
    if en and efi:
        q = f"Translate this to Efik: {en}"
        th = f"Language: Efik.\nIntent: Translate English sentence to Efik.\nInput: '{en}'.\nEfik grammatical rendering."
        all_formatted_texts.append({"text": chatml_template.format(system=MASTER_SYSTEM_PROMPT, user=q, thought=th, assistant=efi)})

# 4. Format Edo
print("🔄 Formatting Edo dataset...")
for item in edo_pairs[:8000]:
    en = item.get("en", "").strip()
    edo = item.get("edo", "").strip()
    if en and edo:
        q = f"How do you say '{en}' in Edo (Bini)?"
        th = f"Language: Edo (Bini).\nIntent: Translate English sentence to Bini.\nInput: '{en}'.\nBini vocabulary synthesis."
        all_formatted_texts.append({"text": chatml_template.format(system=MASTER_SYSTEM_PROMPT, user=q, thought=th, assistant=edo)})

random.shuffle(all_formatted_texts)
final_dataset = Dataset.from_list(all_formatted_texts)
print(f"\n🎉 Total CoT Training Examples Created: {len(final_dataset)}")

## 🧠 Step 5: Load Qwen-2.5-3B-Instruct & LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
load_in_4bit = True

print("📥 Loading Qwen2.5-3B-Instruct via Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 64,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("✅ Qwen2.5-3B LoRA configuration initialized successfully.")

## 🚀 Step 6: Train with Google Drive Auto-Save (500 Steps — ~40 Mins)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

drive_checkpoint_dir = "/content/drive/MyDrive/afriwise_checkpoints"
os.makedirs(drive_checkpoint_dir, exist_ok=True)

training_args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 30,
    max_steps = 500, # Fast 40-minute training (converged at loss ~0.39)
    learning_rate = 2e-4,
    fp16 = True,
    bf16 = False,
    logging_steps = 10,
    save_steps = 100, # Auto-saves checkpoint to Drive every 100 steps
    save_total_limit = 2,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    seed = 3407,
    output_dir = drive_checkpoint_dir,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = final_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = training_args,
)

print("🚀 Starting fast 500-step training (Estimated ~40 minutes)...")
trainer_stats = trainer.train()
print("🎉 Training Complete!")

## 💾 Step 7: Export to GGUF & Save to Google Drive Root

In [ ]:
import shutil

print("🔄 Converting and quantizing to GGUF (Q4_K_M)...")
model.save_pretrained_gguf("afriwise_qwen_gguf", tokenizer, quantization_method = "q4_k_m")

dest_gguf = "/content/drive/MyDrive/afriwise_qwen2.5_3b.Q4_K_M.gguf"

# Search inside output folder for generated GGUF
for root, dirs, files in os.walk("afriwise_qwen_gguf"):
    for f in files:
        if f.endswith(".gguf"):
            src_path = os.path.join(root, f)
            print(f"🚚 Copying {f} directly to Google Drive...")
            shutil.copy(src_path, dest_gguf)
            print(f"✅ Model successfully saved to Google Drive: {dest_gguf}")

print("🎉 Done! Download 'afriwise_qwen2.5_3b.Q4_K_M.gguf' directly from drive.google.com")